# 🎨 ComfyUI trên Google Colab (Free) — WAI-illustrious + YOLO + SAM

**Cách dùng:**
1. `Runtime` → `Change runtime type` → **T4 GPU** → Save
2. **Dán Civitai API key** vào ô `CIVITAI_KEY` ở Cell 2 (civitai.com → Account Settings → API Keys)
3. Chạy lần lượt **Cell 1 → 2 → 3**
4. Copy link Cell 3 → **dán vào thanh địa chỉ tab mới** (đừng bấm trực tiếp)
5. Kéo thả `WAI_ChiTietNho.json` vào ComfyUI → nhấn **R** → node NẠP MODEL chọn **WAI-illustrious** → Queue

**Đã có trong bản này (sửa lỗi node đỏ FaceDetailer / YOLO / SAM):**
- 🥇 Model: **WAI-illustrious**
- 🎯 **ComfyUI-Impact-Pack + Impact-Subpack** (FaceDetailer, HandDetailer)
- 📦 Tải sẵn: `face_yolov8m.pt`, `hand_yolov8s.pt`, `sam_vit_b_01ec64.pth`
- 💾 Checkpoint + YOLO + SAM lưu Google Drive → lần sau không tải lại (~3 phút khởi động)
- 🛡️ Drive lỗi vẫn chạy ổ tạm

⚠️ Drive trống ~8GB (model 6.5GB + SAM 375MB + YOLO ~80MB). Lần đầu ~10–15 phút.

In [ ]:
# ===== CELL 1: Drive + ComfyUI + Impact Pack (YOLO/SAM/FaceDetailer) =====
!nvidia-smi --query-gpu=name,memory.total --format=csv

import os

USE_DRIVE = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    os.makedirs('/content/drive/MyDrive/AI_Models/checkpoints', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/AI_Models/ultralytics/bbox', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/AI_Models/sams', exist_ok=True)
    USE_DRIVE = True
    print('✅ Drive OK — model + YOLO + SAM sẽ được lưu vĩnh viễn.')
except Exception as e:
    print('⚠️ Drive lỗi:', e)
    print('→ Chạy tiếp bằng ổ tạm. Cách sửa Drive: xem ghi chú cuối notebook.')

ROOT = '/content/drive/MyDrive/AI_Models' if USE_DRIVE else '/content/AI_Models'
os.makedirs(f'{ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{ROOT}/ultralytics/bbox', exist_ok=True)
os.makedirs(f'{ROOT}/sams', exist_ok=True)

CKPT_DIR = f'{ROOT}/checkpoints'
YOLO_DIR = f'{ROOT}/ultralytics/bbox'
SAM_DIR  = f'{ROOT}/sams'
open('/content/ckpt_dir.txt', 'w').write(CKPT_DIR)
open('/content/yolo_dir.txt', 'w').write(YOLO_DIR)
open('/content/sam_dir.txt',  'w').write(SAM_DIR)

# 2) Cài ComfyUI
os.chdir('/content')
!rm -rf /content/ComfyUI
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
os.chdir('/content/ComfyUI')

# Giữ PyTorch-CUDA của Colab — không để pip đè bản CPU
!grep -viE '^(torch|torchvision|torchaudio)([=<>!~ ]|$)' requirements.txt > /content/req_notorch.txt
!pip install -q -r /content/req_notorch.txt

import torch
assert torch.cuda.is_available(), '❌ PyTorch không thấy GPU! Runtime → Change runtime type → T4 GPU, Restart session rồi chạy lại Cell 1'
print('✅ PyTorch', torch.__version__, '- GPU:', torch.cuda.get_device_name(0))

# 3) Impact Pack + Subpack (FaceDetailer / YOLO / SAM)
os.chdir('/content/ComfyUI/custom_nodes')
!git clone --depth 1 https://github.com/ltdrdata/ComfyUI-Impact-Pack.git
!git clone --depth 1 https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git

!grep -viE '^(torch|torchvision|torchaudio)([=<>!~ ]|$)' /content/ComfyUI/custom_nodes/ComfyUI-Impact-Pack/requirements.txt > /content/req_impact.txt || true
!grep -viE '^(torch|torchvision|torchaudio)([=<>!~ ]|$)' /content/ComfyUI/custom_nodes/ComfyUI-Impact-Subpack/requirements.txt > /content/req_subpack.txt || true
!pip install -q -r /content/req_impact.txt
!pip install -q -r /content/req_subpack.txt
!pip install -q ultralytics opencv-python-headless scikit-image piexif segment-anything dill

# 4) Trỏ thư mục model sang Drive (hoặc ổ tạm AI_Models)
os.chdir('/content/ComfyUI')
for name, dest in [
    ('models/checkpoints', CKPT_DIR),
    ('models/ultralytics', f'{ROOT}/ultralytics'),
    ('models/sams', SAM_DIR),
]:
    path = f'/content/ComfyUI/{name}'
    !rm -rf {path}
    os.makedirs(os.path.dirname(path), exist_ok=True)
    os.symlink(dest, path)
    print('🔗', path, '→', dest)

print()
print('✅ Xong Cell 1')
print('   Checkpoint:', CKPT_DIR)
print('   YOLO:      ', YOLO_DIR)
print('   SAM:       ', SAM_DIR)
!ls /content/ComfyUI/custom_nodes | grep -i impact

In [ ]:
# ===== CELL 2: Tải WAI-illustrious + YOLO mặt/tay + SAM =====
CIVITAI_KEY = ""  # @param {type:"string"}
# ↑ Dán key vào ô bên phải. Lấy key: civitai.com → Account Settings → API Keys

import os, urllib.request

CKPT_DIR = open('/content/ckpt_dir.txt').read().strip()
YOLO_DIR = open('/content/yolo_dir.txt').read().strip()
SAM_DIR  = open('/content/sam_dir.txt').read().strip()
os.makedirs(YOLO_DIR, exist_ok=True)
os.makedirs(SAM_DIR, exist_ok=True)
print('Checkpoint:', CKPT_DIR)
print('YOLO:      ', YOLO_DIR)
print('SAM:       ', SAM_DIR)

def download(url, path, min_bytes, label):
    if os.path.exists(path) and os.path.getsize(path) > min_bytes:
        print(f'✅ {label} đã có ({os.path.getsize(path)/1e6:.0f} MB) — bỏ qua.')
        return
    print(f'⬇️  Đang tải {label} ...')
    !wget -c -O "{path}" "{url}"
    size = os.path.getsize(path) if os.path.exists(path) else 0
    if size < min_bytes:
        if os.path.exists(path):
            os.remove(path)
        raise RuntimeError(f'❌ Tải {label} thất bại (file {size} byte). Chạy lại cell này.')
    print(f'✅ {label} xong ({size/1e6:.0f} MB)')

# 1) WAI-illustrious (~6.5 GB) — cần Civitai key
WAI = f'{CKPT_DIR}/WAI-illustrious.safetensors'
if os.path.exists(WAI) and os.path.getsize(WAI) > 6_000_000_000:
    print('✅ WAI-illustrious đã có sẵn — bỏ qua tải.')
else:
    assert CIVITAI_KEY.strip(), '❌ Chưa dán Civitai API key vào ô CIVITAI_KEY!'
    download(
        f'https://civitai.com/api/download/models/2514310?token={CIVITAI_KEY.strip()}',
        WAI, 6_000_000_000, 'WAI-illustrious')

# 2) YOLO — HuggingFace Bingsu/adetailer (không cần key)
download(
    'https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt',
    f'{YOLO_DIR}/face_yolov8m.pt', 20_000_000, 'YOLO mặt (face_yolov8m)')
download(
    'https://huggingface.co/Bingsu/adetailer/resolve/main/hand_yolov8s.pt',
    f'{YOLO_DIR}/hand_yolov8s.pt', 8_000_000, 'YOLO tay (hand_yolov8s)')

# 3) SAM ViT-B (~375 MB) — Facebook / HF mirror
SAM = f'{SAM_DIR}/sam_vit_b_01ec64.pth'
try:
    download(
        'https://huggingface.co/flax-community/segment-anything/resolve/main/sam_vit_b_01ec64.pth',
        SAM, 300_000_000, 'SAM ViT-B')
except Exception as e:
    print('⚠️ Mirror HF lỗi, thử Facebook:', e)
    download(
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        SAM, 300_000_000, 'SAM ViT-B (facebook)')

print()
print('----- checkpoints -----')
!ls -lh {CKPT_DIR}
print('----- ultralytics/bbox (phải thấy bbox/face_yolov8m.pt trong ComfyUI) -----')
!ls -lh {YOLO_DIR}
print('----- sams -----')
!ls -lh {SAM_DIR}
print()
print('✅ Model + YOLO + SAM sẵn sàng!')
print('   Trong ComfyUI dropdown YOLO chọn:  bbox/face_yolov8m.pt  và  bbox/hand_yolov8s.pt')
print('   SAMLoader chọn:  sam_vit_b_01ec64.pth')

In [ ]:
# ===== CELL 3: Khởi chạy ComfyUI (nền) + link cloudflared =====
import subprocess, time, socket, re, os

!pkill -f "python main.py" 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(2)

if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

os.chdir('/content/ComfyUI')
comfy_log = open('/content/comfyui.log', 'w')
comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--enable-cors-header'],
    stdout=comfy_log, stderr=subprocess.STDOUT)
print('⏳ Đang khởi động ComfyUI (30–90 giây, Impact Pack load YOLO lần đầu hơi lâu)...')

for _ in range(180):
    time.sleep(1)
    if comfy.poll() is not None:
        raise RuntimeError('❌ ComfyUI bị tắt! Xem lỗi: !tail -40 /content/comfyui.log')
    try:
        with socket.create_connection(('127.0.0.1', 8188), timeout=1):
            break
    except OSError:
        pass
else:
    raise RuntimeError('❌ Quá 3 phút chưa mở cổng. Xem log: !tail -40 /content/comfyui.log')
print('✅ ComfyUI đã chạy!')

cf_log = open('/content/cloudflared.log', 'w')
cf = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188',
     '--http-host-header', '127.0.0.1:8188', '--protocol', 'http2'],
    stdout=cf_log, stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    time.sleep(1)
    txt = open('/content/cloudflared.log').read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
    if m:
        url = m.group(0)
        break
print()
print('=' * 60)
if url:
    print('🎨 COPY LINK NÀY, DÁN VÀO THANH ĐỊA CHỈ TAB MỚI:')
    print(url)
else:
    print('⚠️ Chưa lấy được link cloudflare — dùng Cell 5 (localtunnel)')
print('=' * 60)
print('\n💡 Kéo thả WAI_ChiTietNho.json → R (refresh) → chọn WAI-illustrious → Queue')
print('   Node YOLO/SAM/FaceDetailer phải MÀU BÌNH THƯỜNG (không đỏ).')
print('   Ảnh: /content/ComfyUI/output — tải về máy trước khi ngắt phiên!')

In [ ]:
# ===== CELL 3B (TÙY CHỌN): Giao diện gọn tiếng Việt =====
# Điều kiện: Cell 3 đã chạy (ComfyUI đang chạy nền).
!pip install -q gradio websocket-client
!wget -q -O /content/giaodien_tao_anh.py https://raw.githubusercontent.com/caone1196-sketch/t-i-li-u/arena/01a09a8b-t-i-li-u/giaodien_tao_anh.py
print('⏳ Đợi link https://xxxx.gradio.live hiện ra (~20 giây)')
print('   Cell này CHẠY LIÊN TỤC để giữ giao diện — đừng dừng khi đang dùng.\n')
!python /content/giaodien_tao_anh.py

In [ ]:
# ===== CELL 4: KIỂM TRA sức khỏe + YOLO/SAM/Impact =====
!curl -s -o /dev/null -w "A) ComfyUI nội bộ: HTTP %{http_code} (200 = OK)\n" --max-time 20 http://127.0.0.1:8188/system_stats

import re, os, glob
txt = open('/content/cloudflared.log').read() if os.path.exists('/content/cloudflared.log') else ''
m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
if m:
    url = m.group(0)
    print('   Link hiện tại:', url)
    !curl -s -o /dev/null -w "B) Qua tunnel:     HTTP %{http_code} (200 = OK, 502 = ComfyUI chết)\n" --max-time 40 {url}/system_stats
else:
    print('B) Không tìm thấy link cloudflared')

print('\n----- custom nodes Impact -----')
!ls /content/ComfyUI/custom_nodes | grep -i impact || echo '❌ THIẾU Impact Pack — chạy lại Cell 1'

print('\n----- YOLO (cần face_yolov8m + hand_yolov8s) -----')
!ls -lh /content/ComfyUI/models/ultralytics/bbox/ 2>/dev/null || echo '❌ THIẾU thư mục ultralytics/bbox — chạy lại Cell 1–2'

print('\n----- SAM -----')
!ls -lh /content/ComfyUI/models/sams/ 2>/dev/null || echo '❌ THIẾU SAM — chạy lại Cell 2'

print('\n----- log ComfyUI (lỗi import Impact sẽ hiện ở đây) -----')
!tail -40 /content/comfyui.log

In [ ]:
# ===== CELL 5 (DỰ PHÒNG): Link localtunnel =====
!npm install -g localtunnel > /dev/null 2>&1

import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
print('=' * 60)
print('🔑 MẬT KHẨU (Tunnel Password) khi trang hỏi:', ip)
print('=' * 60)
print('Đợi link https://....loca.lt hiện ra rồi mở.\n')
!lt --port 8188

## 📝 Ghi chú

**Node đỏ FaceDetailer / UltralyticsDetectorProvider / SAMLoader:** notebook cũ chưa cài Impact Pack. Dùng file notebook **mới này**, chạy lại từ Cell 1. Trong ComfyUI nhấn **R**. Dropdown YOLO phải có `bbox/face_yolov8m.pt` và `bbox/hand_yolov8s.pt`; SAMLoader có `sam_vit_b_01ec64.pth`.

**Workflow khuyên dùng:** `WAI_ChiTietNho.json` (3 lượt + cắt vẽ mắt/tay). Không có pack thì dùng `WAI_3Luot.json` (chỉ node gốc, không đỏ).

**Khởi động phiên mới:** Cell 1 → 2 (bỏ qua nếu đã có trong Drive) → 3. ~3 phút.

**Chọn model:** nhấn **R** → node NẠP MODEL → `WAI-illustrious.safetensors`. Ctrl+S lưu workflow.

**Thông số WAI:** cfg 5, euler_ancestral, 832×1216, tag `masterpiece, best quality, newest, absurdres, highres`.

**Lưu ảnh:** `/content/ComfyUI/output` mất khi ngắt phiên. Chuột phải → Save image, hoặc `!zip -r /content/anh.zip /content/ComfyUI/output`.

**Sự cố:**
- Không GPU → Runtime → T4 GPU
- Link 403 → copy dán tab mới
- 502 → Cell 4 xem log → chạy lại Cell 3
- Impact import error trong log → Cell 1 pip chưa xong, Restart session → Cell 1
- YOLO dropdown trống → Cell 2 chưa tải, hoặc symlink `models/ultralytics` gãy (Cell 4 kiểm tra)
- `Torch not compiled with CUDA` → Disconnect and delete runtime → Cell 1
- ⚠️ Đừng chia sẻ notebook đã dán Civitai key

## 🔧 Drive "mount failed"
1. Runtime → Disconnect and delete runtime → Cell 1
2. Cửa sổ quyền: Cho phép tất cả
3. Cho popup + cookie bên thứ ba với colab.research.google.com
4. Cửa sổ ẩn danh, 1 Gmail cá nhân